# Possession Dataset — EDA

Exploratory analysis of the processed possession-level dataset.
Includes:
- Label distribution
- Feature distributions by label
- Competition-level breakdown
- Sequence length analysis
- Play pattern analysis

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / 'data' / 'processed'
SPLITS = PROCESSED / 'splits'

df = pd.read_parquet(PROCESSED / 'possessions' / 'possessions.parquet')
print(f'Total possessions: {len(df):,}')
print(f'Columns: {list(df.columns)}')

## 1. Label Distribution

In [ ]:
shot_rate = df['ends_in_shot'].mean()
print(f'Shot possessions  : {df["ends_in_shot"].sum():,}  ({shot_rate:.1%})')
print(f'No-shot possessions: {(1 - df["ends_in_shot"]).sum():,}  ({1-shot_rate:.1%})')

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['No Shot', 'Shot'], df['ends_in_shot'].value_counts().sort_index(), color=['#64748b','#ef4444'])
ax.set_title('Label Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(df['ends_in_shot'].value_counts().sort_index()):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'trained' / 'eda_label_dist.png', dpi=120)
plt.show()

## 2. Shot Rate by Competition

In [ ]:
comp_stats = df.groupby('competition_label').agg(
    possessions=('ends_in_shot', 'count'),
    shot_rate=('ends_in_shot', 'mean'),
    matches=('match_id', 'nunique')
).reset_index()
print(comp_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(comp_stats['competition_label'], comp_stats['shot_rate']*100, color='#3b82f6', alpha=0.8)
ax.set_ylabel('Shot Possession Rate (%)')
ax.set_title('Shot Rate per Competition')
for bar, rate in zip(bars, comp_stats['shot_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{rate:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'trained' / 'eda_shot_rate_comp.png', dpi=120)
plt.show()

## 3. Sequence Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, ax in zip([0, 1], ['#64748b','#ef4444'], axes):
    subset = df[df['ends_in_shot'] == label]['n_events']
    ax.hist(subset.clip(upper=40), bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'Sequence Length — {"Shot" if label else "No Shot"} (median={subset.median():.0f})')
    ax.set_xlabel('Events per possession')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig(ROOT / 'models' / 'trained' / 'eda_seq_length.png', dpi=120)
plt.show()

## 4. Territorial Progression

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for feat, ax in zip(['start_x', 'end_x', 'progression'], axes):
    for label, color in [(0, '#64748b'), (1, '#ef4444')]:
        subset = df[df['ends_in_shot'] == label][feat]
        ax.hist(subset, bins=40, color=color, alpha=0.6, label='Shot' if label else 'No Shot', density=True)
    ax.set_title(feat)
    ax.set_xlabel('Field position (0-120)')
    ax.legend()

plt.tight_layout()
plt.savefig(ROOT / 'models' / 'trained' / 'eda_territory.png', dpi=120)
plt.show()

## 5. Play Pattern Analysis

In [ ]:
pp_stats = df.groupby('play_pattern').agg(
    count=('ends_in_shot', 'count'),
    shot_rate=('ends_in_shot', 'mean')
).sort_values('shot_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444' if r > 0.15 else '#3b82f6' if r > 0.08 else '#94a3b8' for r in pp_stats['shot_rate']]
bars = ax.barh(pp_stats.index, pp_stats['shot_rate'] * 100, color=colors)
ax.set_xlabel('Shot Rate (%)')
ax.set_title('Shot Rate by Play Pattern')
for bar, (_, row) in zip(bars, pp_stats.iterrows()):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{row["shot_rate"]:.1%} (n={row["count"]:,})', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / 'models' / 'trained' / 'eda_play_pattern.png', dpi=120)
plt.show()

## 6. Train/Val/Test Split Overview

In [ ]:
for split in ['train', 'validation', 'test']:
    match_ids = pd.read_csv(SPLITS / f'{split}_matches.csv')['match_id'].tolist()
    subset = df[df['match_id'].isin(match_ids)]
    print(f'{split:12s}: {len(match_ids):3d} matches | {len(subset):6,} possessions | shot_rate={subset["ends_in_shot"].mean():.1%}')
    comp_dist = subset['competition_label'].value_counts().to_dict()
    for comp, cnt in comp_dist.items():
        print(f'             {comp}: {cnt:,} possessions')
    print()